# Calibrated intensity servo: two sensor modes, feedforward, PI/PII/PIII, and shot records

This notebook controls **ADC224/1 → intensity controller → DAC228/3**. Keep your existing 80 MHz AOM carrier and calibrated 90 MHz mixer LO/demodulator settings. It does not initialize clocks, reprogram RFDC mixers, or configure either laser PLL.

Before first use, load the matching IS03 `.bit` and `.hwh` with your normal RFSoC clock/RFDC initialization. The first code cell below only attaches to hardware already loaded; it cannot upgrade an older bitstream. After a download, restore/verify the RFDC demod settings, mixer LO amplitude (`axi_gpio_12`), and demodulator phase correction (`axi_gpio_10`) used for calibration; use the supplied `intensity_phase_calibration.ipynb` if needed. Match the physical PD attenuation to the sensor table. Loading the table does not restore those settings.

The new path uses **optical power**. Low-power mode uses `50 mW = 65536 power codes` (0.763 µW/code), while high-power mode uses `200 mW = 65536 power codes` (3.052 µW/code). The largest uint16 value is one code below full scale. These are numerical steps, not claims of ADC measurement resolution. The original path still uses raw ADC setpoint counts. **Old ramp words cannot be reused unchanged in the new path.** Two sensor tables correct the fixed PD attenuation modes. A common sin² inverse corrects RF-amplitude-to-optical-power conversion after adding feedforward and feedback.

Work through the sections deliberately. Table loading/controller changes are done at zero power between shots. The measured plant has about **1.087 µs effective delay**, including the INT2 receive filter; controller predictions add the new 44.76 ns digital path (52.90 ns with the shelf). The user has already achieved above 300 kHz with PI, so the selectable PI starting trial now targets **300 kHz crossover**. PII/PIII default to **150 kHz** with wider predicted gain margins. The supplied large-dither FRA predicts only about 0.166 dB gain margin for the 300 kHz PI trial despite 48.1° local phase margin; remeasure at small amplitude before automatic tuning. This is a provisional model, not a contradiction of the observed PI performance.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pynq import Overlay
from IPython.display import display, Markdown

import intensity_servo as servo
import intensity_calibration as intensity
import intensity_shots as shots
import intensity_shot_analysis as analysis
import tp_intensity_rfsoc_buffer as tp
for module in (servo, intensity, shots, analysis, tp):
    importlib.reload(module)

# Attach only if this notebook does not already have the correct loaded overlay.
# This does NOT download a bitstream or reset a running experiment.
bit_path = '/home/xilinx/Bitfiles/FreqLocksBranch/overlays/TransportPhaseLockFF.bit'
if 'ol' not in globals():
    ol = Overlay(bit_path, download=False)
servo.check(ol)
shots.check(ol)
print('Servo helper:', servo.__file__)
print('FPGA servo signature:', hex(ol.intensity_servo_read.read(8)))
print(servo.status(ol))

output_dir = Path('intensity_servo_runs') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
output_dir.mkdir(parents=True, exist_ok=True)
print('Results directory:', output_dir.resolve())

## 1. Load the measured calibration files in Python

Copy the supplied `exports/intensity_servo_20261028` directory beside this notebook. The user confirmed the table reaching 167.6 mW is **high power**, and both measurements used ADC bias zero. The 18 µW two-point file is excluded. Select the software mode to match the physical fixed PD attenuation switch.

The following cell only computes tables in RAM. Each sensor table has 16384 entries; the AOM inverse has 65536 entries. These become direct block-RAM lookups in the FPGA, not run-time arctangent/sine calculations. The fitted AOM response is `F(u)=sin(alpha*u)**2`; `alpha≈4.72156e-5 rad/code`, capped at the largest measured drive, 21335 codes.

In [ ]:
calibration_dir = Path('intensity_servo_20261028')
low_sensor = servo.load_sensor_table(calibration_dir / 'sensor_low_power.json')
high_sensor = servo.load_sensor_table(calibration_dir / 'sensor_high_power.json')
aom_model = json.loads((calibration_dir / 'aom_model.json').read_text())
# Two numerical ranges; both waveforms are specified in physical mW.
full_scale_by_mode_mW = {'low_power': 50.0, 'high_power': 200.0}
sensor_luts = servo.make_sensor_luts(low_sensor, high_sensor,
    power_full_scale_mW=full_scale_by_mode_mW['high_power'],
    low_power_full_scale_mW=full_scale_by_mode_mW['low_power'])
output_lut = servo.make_output_lut(
    alpha_rad_per_code=aom_model['alpha_rad_per_code'],
    maximum_command_codes=21335)
for label, table in [('low power', low_sensor), ('high power', high_sensor)]:
    plt.plot(table['adc_counts'], table['optical_power'], '.-', label=label)
plt.xlabel('Raw ADC I [counts]'); plt.ylabel('Optical power [mW]')
plt.grid(); plt.legend(); plt.show()
print('Numerical step [µW/code]:', {m: s*1000/65536 for m,s in full_scale_by_mode_mW.items()})
print('Measured ADC ranges:', sensor_luts['adc_limits_counts'])
print('Measured AOM cap:', output_lut['maximum_command_codes'], 'codes')
simulation_notes = calibration_dir / 'tracking_simulation.md'
if simulation_notes.exists():
    display(Markdown(f'[Feedforward tracking simulation and assumptions]({simulation_notes.as_posix()})'))
simulation_plot = calibration_dir / 'tracking_simulation.png'
if simulation_plot.exists():
    display(Markdown(f'[Feedforward tracking simulation plots]({simulation_plot.as_posix()})'))

## 2. Cold initialization / deliberate between-shot reconfiguration

This cell **turns the intensity output off**, loads the three tables, and initializes the new controller. Do not run it during an experimental shot. Close competing notebook/server owners of the same command DMA first; an armed shot is rejected.

Choose a preset explicitly. `PI` targets 300 kHz with a 50 kHz integral corner; `PII` and `PIII` target 150 kHz with a 400 kHz main corner, then 1 kHz and optionally 100 Hz slow corners. `PI_150k_fallback` gives the wider-margin one-integrator alternative. `PI_lead_300k` uses a 300 kHz PI corner and a 20/200 kHz lead shelf; the model improves gain margin to only about 0.64 dB in the refined model, so it also needs verification. Presets are estimates for the measured operating point, not validated hardware settings. No preset changes the RF carrier frequency.

Feedforward supplies the calibrated static power command immediately; feedback corrects residual error. It can strongly improve smooth ramps, but it is **not universally better**: with the same feedback gains it can excite sharper transients and increase RMS error on abrupt steps. Ordinary static feedforward does not cancel acoustic propagation delay and does not change the small-signal feedback poles at a fixed operating point. The initial step waveform therefore defaults to **feedforward off** for the 300 kHz PI bring-up. Set `feedforward_enabled=True` for a separate comparison run, holding the waveform, feedback coefficients and optical scale fixed. Do not mix feedforward-on/off records within one optimizer trial set.

The provisional 150 kHz PI simulation gives smooth-shot RMS **113.2 → 7.45 µW** with feedforward (about 15× lower), but mixed abrupt-shot RMS **0.620 → 0.741 mW** (about 20% higher). Those are modeled whole-shot errors, not hardware measurements or the optimizer's illuminated-sample score. A +5% optical-scale error can drive the narrow-margin 300 kHz PI case into railing. Review the linked simulation assumptions and check real traces before enabling feedforward or gain adaptation.

`effective_power_mW` is the optical scale in `P=A_eff*sin²(alpha*u)`. The supplied FRA gives approximately 184 mW at its operating point, but this is **not a separately measured absolute AOM input power**. It includes diffraction efficiency and collection losses. Use a fresh settled power/drive observation if the incident beam or optical path changed. The same inverse scale acts on feedforward and feedback once.

The [full return-ratio plots](intensity_servo_20261028/open_loop_predictions.md) include relative pipeline registers. The PI preset has only about **0.2 dB gain margin**; its almost-flat unity-gain region makes the predicted crossing sensitive (about 264 kHz with the refined model versus 299 kHz with the scalar preset estimate). PII/PIII predict about 149 kHz, 48° phase margin and 5.1 dB gain margin. Do not enable the shelf at unchanged Kp: its high-frequency gain is 10×. These are model predictions, not measured performance of the new bitstream.


In [ ]:
sensor_mode = 'low_power'        # Match the physical PD attenuation.
power_full_scale_mW = full_scale_by_mode_mW[sensor_mode]
power_per_code = power_full_scale_mW/65536
controller_preset = 'PI'  # PI, PII, PIII, PI_150k_fallback, or PI_lead_300k.
feedforward_enabled = False  # Initial PI300k step bring-up; compare True in a separate run.
starting_points = json.loads((calibration_dir / 'controller_starting_points.json').read_text())
preset = starting_points[controller_preset]
controller_kind = controller_preset.split('_')[0]
effective_power_mW = 184.0  # Conditional initial value inferred from the supplied FRA.
coefficients = servo.design_controller(
    controller_kind, kp=preset['proportional_gain'],
    integral_corners_hz=preset['integrator_corner_hz'],
    lead_zero_hz=preset['lead_zero_hz'], lead_pole_hz=preset['lead_pole_hz'])
print('Predicted margins:', preset['margins_in_measured_band'])

servo.safe_off(ol)
servo.manual_power(ol, 0, power_full_scale_mW=power_full_scale_mW)
servo.load_calibration(ol, sensor_luts, output_lut)
servo.configure_controller(ol, coefficients, sensor_mode=sensor_mode,
                           feedforward=feedforward_enabled, feedback=True)
servo.set_incident_power(ol, effective_power_mW)
servo.enable(ol)                 # Still zero reference; releases manual DAC override.
configuration = servo.configuration(ol)
(output_dir / 'servo_configuration_initial.json').write_text(json.dumps(configuration, indent=2))
print('Configured at zero power:', servo.status(ol))

## 3. Establish one stable operating point

Select a power comfortably inside the selected sensor range and available AOM range. Low-power calibration covers 0.092–46.98 mW; high-power covers 1.814–167.62 mW. The fitted drive cap may impose a lower available maximum for the current `A_eff`. A nonzero reference at a physical sensor attenuation different from the selected software mode is not a valid operating point.

Use `servo.manual_power`, **not** the old `intensity.manual_setpoint` ADC-count helper, in calibrated mode. To switch attenuation, first set zero, call `safe_off`, change the physical attenuation and `sensor_mode`, then reconfigure/enable at zero. The helper switches the output scale with the sensor mode; regenerate any packed waveforms using that mode’s full scale. Physical-mW uploads are converted automatically.

In [ ]:
operating_power_mW = 10.0
print('Actual quantized setpoint:', servo.manual_power(ol, operating_power_mW), 'mW')
time.sleep(0.5)
print(servo.status(ol))

# A no-dither diagnostic capture uses the existing intensity monitor.
intensity.stop(ol)
record = intensity.capture(ol, samples=131072, decimation_log2=4)
i_counts, controller_command, dac_command, dds_phase = intensity.unpack(record)
sensor = low_sensor if sensor_mode == 'low_power' else high_sensor
measured_mW = intensity.linearize(i_counts, sensor)
t_ms = np.arange(len(measured_mW)) / record['sample_rate_hz'] * 1000
fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
ax[0].plot(t_ms, measured_mW); ax[0].set_ylabel('Measured power [mW]')
ax[1].plot(t_ms, dac_command); ax[1].set_ylabel('RF amplitude [DAC codes]')
ax[1].set_xlabel('Time [ms]')
for a in ax: a.grid()
plt.show()
print('Diagnostic flags:', [name for bit, name in intensity.FAULT_NAMES.items()
                            if record['status'] & (1 << bit)])

## 4. FRA of the new loop, with the existing averaged sine measurement

The dither is still added **after the AOM inverse**, in raw DAC-amplitude codes. `L=-C/U` is the local return ratio; `K=Q/(P0*U)` remains fractional optical response per raw DAC code. Use the raw sensor table corresponding to the selected attenuation. The old HLS PI is disabled; the updated helper recognizes the enabled calibrated controller.

Begin with a single point and check flags, measured modulation fraction and repeatability. The old 500-code measurement reached 50% optical modulation near resonance, so reliability flags alone did not establish small-signal behavior. Compare half-amplitude measurements before final tuning. A smaller dither and 1 kHz lower sweep limit reduce saturation and execution time. The helper prints after completing each frequency's repeats; Ctrl-C ramps off diagnostic dither and leaves the feedback controller running.

In [ ]:
point = intensity.measure_point(ol, 10e3, amplitude_codes=32, table=sensor,
                                minimum_measure_s=0.02, cycles=16,
                                raw_path=output_dir / 'first_fra_point.npz')
print({key: point[key] for key in ('frequency_hz', 'fractional_modulation',
                                  'flags', 'loop', 'K')})

frequencies_hz = np.geomspace(1e3, 1e6, 31)
fra_csv = output_dir / f'intensity_loop_{datetime.now():%H%M%S_%f}.csv'
rows = intensity.sweep(
    ol, frequencies_hz, amplitude_codes=32, table=sensor,
    repeats=4, max_repeats=8, target_snr=5,
    maximum_drift_fraction=0.03, minimum_measure_s=0.02,
    csv_path=fra_csv)
print('Saved:', fra_csv.resolve())

In [ ]:
fra_data = pd.read_csv(fra_csv)
f_hz = fra_data.frequency_hz.to_numpy()
L = fra_data.loop_re.to_numpy() + 1j*fra_data.loop_im.to_numpy()
fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax[0].semilogx(f_hz, 20*np.log10(abs(L)), '.-'); ax[0].axhline(0, color='k', lw=.7)
ax[0].set_ylabel('Loop magnitude [dB]')
ax[1].semilogx(f_hz, np.unwrap(np.angle(L))*180/np.pi, '.-')
ax[1].axhline(-180, color='k', lw=.7); ax[1].set_ylabel('Loop phase [deg]')
ax[1].set_xlabel('Frequency [Hz]')
for a in ax: a.grid(which='both', alpha=.3)
plt.show()

## 5. Prepare a physical-power waveform and per-shot recorder

The existing dwell streamer still owns waveform timing and its existing external hardware trigger. Each 32-bit word is `[power_code:16, dwell_minus_one:16]`. The helper splits long holds and appends the existing one-clock zero/TLAST ending. A new independent receive DMA captures **reference waveform, measured power, error, raw ADC and actual actuator output** from each triggered shot; it adds no feedback delay.

Only one process may own `axi_dma_6`. The example below creates a Jupyter owner; do not create it concurrently with a running ZMQ server. In the server workflow pass the same `shot_recording` dictionary to the updated server/buffer instead. For BLACS use the explicit `set_power_schedule(time_s, power_mW, sensor_mode=...)` upload described in `docs/intensity_power_upload.md`. The server converts mW using the selected mode, before the trigger. The old compiled NI AO trace already contains converted voltages; do not relabel it mW or run it through a second calibration. `TP_INT_SCALE` applies only to that legacy path.

D=6 records at 3.84 MS/s and approximately 61.4 MB/s during the shot. For the 100 ms example below that is about 6.2 MB. Set D=4 for a shorter capture when resolving faster dynamics matters. Longer shots need an explicitly chosen larger D or a larger memory budget. Recording uses point decimation, **not anti-alias averaging**; do not interpret arbitrary low-rate captures as calibrated broadband PSDs.

In [ ]:
servo.manual_power(ol, 0)
intensity.stop(ol)

# Values are held between adjacent time points; final value is only the endpoint.
waveform_time_s = np.array([0, .005, .030, .055, .080, .100])
waveform_power_mW = np.array([0, 5, 10, 15, 0, 0])
words = servo.waveform_words(waveform_time_s, waveform_power_mW,
                             power_full_scale_mW=power_full_scale_mW)
shot_recording = dict(
    directory=str(output_dir / 'shots'),
    decimation_log2=6, tail_s=.001, maximum_buffer_bytes=64*1024**2,
    metadata={'power_units': 'mW'},
)
buffer = tp.TpIntensityBufferController(
    overlay=ol, max_words=max(10000, len(words)),
    shot_recording=shot_recording)
print('Waveform commands:', len(words))
print('Recorder plan:', shots.capture_plan(waveform_time_s[-1],
                                           decimation_log2=6, tail_s=.001))

Run the next cell once to arm. Then send the normal experimental hardware trigger. `load_power_and_arm` converts the waveform and starts the command DMA but does not itself generate a trigger. `wait_done` saves the record after the actual shot plus tail. Repeat these two cells for each experimental shot. No file is produced merely by drawing a ramp in Python.

A zero-to-positive step naturally starts below the measured sensor range. The analysis allows a 10 µs startup window for that specific below-range condition and at most one ambiguous boundary frame, retains the original flags, excludes those samples from the measured RMS score, and rejects the record if excluded startup exceeds 0.1% of the illuminated samples. A persistent range fault, an above-range sample or another fault is not accepted as normal startup; review the reported excluded startup duration and raw trace.

In [ ]:
buffer.select_serial_setpoint()
print(buffer.load_power_and_arm(waveform_time_s, waveform_power_mW,
    sensor_mode=sensor_mode, metadata={'waveform_units': 'mW',
                                           'feedforward_enabled': bool(feedforward_enabled),
                                           'purpose': 'initial calibrated intensity test'}))

In [ ]:
completion = buffer.wait_done(timeout_s=30.0)
print(completion)
shot_path = Path(completion['shot_record']['path'])
with np.load(shot_path, allow_pickle=False) as saved:
    shot = {key: saved[key] for key in saved.files if key != 'metadata_json'}
    shot_metadata = json.loads(str(saved['metadata_json']))
metrics = analysis.shot_metrics(shot, startup_grace_s=10e-6, maximum_excluded_startup_fraction=0.001)
print(metrics)
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(shot['time_s']*1e3, shot['reference_power'], label='Reference')
ax[0].plot(shot['time_s']*1e3, shot['measured_power'], label='Measured', lw=.7)
ax[0].set_ylabel('Power [mW]'); ax[0].legend()
ax[1].plot(shot['time_s']*1e3, shot['error_power']); ax[1].set_ylabel('Error [mW]')
ax[2].plot(shot['time_s']*1e3, shot['actuator_codes']); ax[2].set_ylabel('Actuator [codes]')
ax[2].set_xlabel('Time after shot start [ms]')
for a in ax: a.grid()
plt.show()
print('Saved shot:', shot_path.resolve())

## 6. Bounded parameter optimization using repeated shots

Keep detector mode, feedforward, the incident-power estimate and the exact physical waveform fixed during optimization. The checks below freeze the feedforward flag within one physical `reference_key` comparison and record its actual hardware state in every trial. For an on/off comparison, start a separate output directory/trial set and compare the saved raw RMS metrics at identical feedback settings. Do not enable Bayesian power adaptation yet. `next_controller_trial` performs a small coordinate search around a stable baseline, requires repeated shots, and rejects faulty/clipped candidates. It does not infer stability from one attractive trace or create triggers.

First review a fresh small-amplitude FRA at the selected operating point. The 300 kHz PI prediction has only about 2% total-gain headroom, so a blind ±10% gain trial is inappropriate. The example uses **0.5% steps**, permits only downward Kp changes from the tested baseline, and keeps controller order, slower corners and the optional shelf fixed. Extend bounds only after measurement establishes margins. This optimizer minimizes the measured RMS optical tracking error in mW over valid illuminated samples, with clipping/range/startup checks acting as acceptance gates. Repeated-shot RMS values are combined by RMS; scatter is reported separately. It does not infer bandwidth or stability margins from an attractive shot trace. Start a new trial log for this RMS-only objective; do not mix earlier normalized or penalized scores.

In [ ]:
# Initialize once for one comparison round; do not reset between trials.
controller_fra_reviewed = False  # Set True only after inspecting new small-signal FRA/margins.
assert controller_fra_reviewed, 'Review the new FRA before automatic parameter trials.'
baseline = {'kp': coefficients['kp_actual'],
            'corner1_hz': coefficients['integral_corners_actual_hz'][0]}
bounds = {'kp': (0.95 * baseline['kp'], baseline['kp']),
          'corner1_hz': (0.8 * baseline['corner1_hz'], 1.2 * baseline['corner1_hz'])}
trials = []
optimization_feedforward_enabled = bool(servo.configuration(ol)['control_word'] & 4)
assert optimization_feedforward_enabled == bool(feedforward_enabled)
reference_key = shot_metadata['reference_key']
trial_log = output_dir / 'controller_trials.json'
print('Initialized a new optimization round; do not rerun this initialization between trials.')

In [ ]:
# Re-run to get the next recommended candidate, then arm the same waveform.
assert bool(feedforward_enabled) == optimization_feedforward_enabled, 'Start a new trial set after changing feedforward.'
assert bool(servo.configuration(ol)['control_word'] & 4) == optimization_feedforward_enabled
assert all(t['feedforward_enabled'] == optimization_feedforward_enabled for t in trials)
proposal = analysis.next_controller_trial(
    baseline, bounds, trials, reference_key=reference_key,
    fraction=.005, repeats=3)
print(proposal)

In [ ]:
# Execute only for action='measure'. This briefly switches the beam off.
if proposal['action'] == 'measure':
    assert bool(feedforward_enabled) == optimization_feedforward_enabled
    trial_parameters = proposal['parameters']
    servo.manual_power(ol, 0)
    servo.safe_off(ol)
    trial_coefficients = servo.design_controller(
        controller_kind, kp=trial_parameters['kp'],
        integral_corners_hz=[trial_parameters['corner1_hz']] +
                            coefficients['integral_corners_actual_hz'][1:],
        lead_zero_hz=coefficients['lead_zero_hz'],
        lead_pole_hz=coefficients['lead_pole_hz'])
    servo.configure_controller(ol, trial_coefficients, sensor_mode=sensor_mode,
                               feedforward=feedforward_enabled, feedback=True)
    servo.enable(ol)
    buffer.select_serial_setpoint()
    print(buffer.load_power_and_arm(waveform_time_s, waveform_power_mW,
    sensor_mode=sensor_mode, metadata={'parameters': trial_parameters,
                                               'feedforward_enabled': optimization_feedforward_enabled,
                                               'purpose': 'bounded controller optimization'}))
# Send the normal hardware trigger before the next cell.

In [ ]:
if proposal['action'] == 'measure':
    completion = buffer.wait_done(timeout_s=30.0)
    trial_path = Path(completion['shot_record']['path'])
    with np.load(trial_path, allow_pickle=False) as saved:
        trial_shot = {key: saved[key] for key in saved.files if key != 'metadata_json'}
        trial_meta = json.loads(str(saved['metadata_json']))
    trial_metrics = analysis.shot_metrics(trial_shot, startup_grace_s=10e-6, maximum_excluded_startup_fraction=0.001)
    trial_feedforward = bool(trial_meta['servo_configuration_start']['control_word'] & 4)
    assert trial_feedforward == optimization_feedforward_enabled, 'Do not mix feedforward modes in one trial set.'
    trials.append(dict(parameters=trial_parameters, score=trial_metrics['score'],
                       feedforward_enabled=trial_feedforward,
                       usable=trial_metrics['usable'], reference_key=trial_meta['reference_key'],
                       raw_file=str(trial_path)))
    trial_log.write_text(json.dumps(trials, indent=2))
    print(trial_metrics)
    print('Saved trial log:', trial_log.resolve())
# Re-run the proposal call (not trials=[]), then the next candidate cells.
# action='complete' returns best_parameters; review records/FRA and apply them
# with the same zero-power configuration sequence before starting a new round.

## 7. Estimate effective input-power drift after controller tuning

For a fixed calibrated AOM curve, `P=A_eff*F(u)` lets settled power/actuator records estimate `A_eff`. This is the effective optical scale at the calibrated sensor plane; it is not an independent absolute incident-power measurement. The scalar Bayesian update tracks `log(A_eff)` with a slow random-walk prior.

Use stationary illuminated plateaus, after transport/settling, and exclude clipped/out-of-range samples. Default estimator commands start at 4000 codes to avoid the measured low-command anomaly. A ≥1% observation floor prevents correlated samples and model error from producing false certainty; above 4000 codes the integrated AOM shape agrees within about 0.8%, while sensor calibration contributes additional uncertainty. Below 4000 the disagreement reaches 19%, so do not simply lower the floor to force updates. At least three complete stationary blocks are needed.

The following cell calculates an observation and estimate **without changing hardware**.

In [ ]:
estimator_table = servo.aom_table(
    alpha_rad_per_code=output_lut['alpha_rad_per_code'],
    minimum_command_codes=4000, maximum_command_codes=21335)
observation = analysis.incident_power_observation(
    shot, estimator_table, block_s=.001, settle_s=.0001,
    delay_s=1.087e-6, minimum_transmission=.03,
    maximum_error_fraction=.03, observation_floor_fraction=.01)
prior = analysis.power_prior(servo.configuration(ol)['effective_power_mW'],
                             fractional_uncertainty=.10)
estimate = analysis.update_power_estimate(
    prior, observation, elapsed_s=1.0, drift_fraction_per_sqrt_s=.01)
print('Observation:', observation)
print('Posterior:', estimate)

## 8. Opt-in automatic updates after each saved shot

Enable this only after selecting and rechecking the optimized controller. It freezes those controller coefficients and updates the **one common inverse-power scale** for both feedforward and feedback. Do not also rescale Kp/Ki by the same factor.

The buffer checks the controller/calibration signature, waits for command and receive DMA completion, requires the shot to end at zero reference/zero actuator, limits each correction, and writes an `.adaptation.json` sidecar with the observation and settings before/after. Its Bayesian state is saved in a separate JSON. A mode/calibration/controller change invalidates certification. Updates happen during the intentionally dark inter-shot interval; they reset the feedback integrator and do not claim a bumpless update while illuminating atoms. The initial correction cap is 0.5% per shot. That limit does not establish safe cumulative gain changes: the fresh FRA must justify the allowed power-scale range, especially for the narrow-margin 300 kHz PI trial.

In [ ]:
optimized_controller_verified = False  # Set True after reviewing optimized shots and new FRA.
controller_fra_reviewed = globals().get('controller_fra_reviewed', False)
if optimized_controller_verified and controller_fra_reviewed:
    current = servo.configuration(ol)
    buffer.shot_recording['adapt_incident_power'] = dict(
        optimizer_certified=True,
        controller_signature=shots.controller_signature(current),
        state_path=str(output_dir / 'incident_power_state.json'),
        nominal_power_mW=current['effective_power_mW'],
        maximum_step_fraction=.005,
        allowed_power_ratio=[.5, 2.0],
        minimum_command_codes=4000,
        observation=dict(block_s=.001, settle_s=.0001, delay_s=1.087e-6,
                         minimum_transmission=.03, maximum_error_fraction=.03,
                         observation_floor_fraction=.01),
    )
    print('Automatic bounded power adaptation enabled after future saved shots.')
else:
    buffer.shot_recording.pop('adapt_incident_power', None)
    print('Automatic adaptation remains off; shot recording still runs.')
# Subsequent buffer.load_power_and_arm(...) / buffer.wait_done(...) pairs record and,
# only when certified and accepted, update the effective-power scale.

## Files and later notebook sessions

The run directory contains the initial configuration, FRA CSVs plus block captures, per-shot NPZ time series, optimizer trial records, and (if enabled) Bayesian state/sidecars. NPZ files retain flags and configuration provenance. Do not compare optimization scores across different physical waveforms or detector calibration modes.

A new attach-only `ol` does not know which table bytes were loaded because the RAM has no readback port. Save `servo.configuration(ol)` at the end of the session. In a new notebook attached to the **same uninterrupted FPGA/table load**, `shots.attach_saved_configuration(ol, saved)` checks the registers and restores host provenance. Matching register counts do not prove identical table contents after reprogramming, so reload the tables after a new bitstream download. Reinitializing clocks or calling `Overlay(..., download=True)` resets the experiment; neither is needed for ordinary record analysis.

In [ ]:
(output_dir / 'servo_configuration_latest.json').write_text(
    json.dumps(servo.configuration(ol), indent=2))
print('Run directory:', output_dir.resolve())

# Explicitly turn off intensity when desired; do not run during a shot:
# servo.manual_power(ol, 0)
# servo.safe_off(ol)